# Homework 9: Supervised Learning with PySpark MLlib

Ryan Mersereau

## Mushroom Edibility Classification

In this notebook, we'll be using the mushroom dataset found at the UCI ML repository [here.](https://archive-beta.ics.uci.edu/dataset/73/mushroom)

We'll try and predict whether a mushroom is **edible (e)** or **poisonous (p)** from 22 categorical features. The full dataset contains 8,124 observations.

## Data Loading & Setup

First we'll import pandas and initialize our `spark` session

In [1]:
import pandas as pd
from pyspark.sql import SparkSession 
spark = SparkSession.builder \
    .appName("MushroomClassification") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 21:15:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/13 21:15:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/13 21:15:24 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/13 21:15:24 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/13 21:15:24 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/04/13 21:15:24 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


Now, lets read in the data using pandas and convert it into a `spark` SQL style dataframe.

The dataset has no column names, so we'll have to add those manually. The first variable `class` is the response, taking values `e` or `p`. Information for the rest of the variables and their single-letter codes is as follows:

Variables Info:
 *    1. cap-shape:                bell=b,conical=c,convex=x,flat=f, knobbed=k,sunken=s
 *    2. cap-surface:              fibrous=f,grooves=g,scaly=y,smooth=s
 *    3. cap-color:                brown=n,buff=b,cinnamon=c,gray=g,green=r, pink=p,purple=u,red=e,white=w,yellow=y
 *    4. bruises?:                 bruises=t,no=f
 *    5. odor:                     almond=a,anise=l,creosote=c,fishy=y,foul=f, musty=m,none=n,pungent=p,spicy=s
 *    6. gill-attachment:          attached=a,descending=d,free=f,notched=n
 *    7. gill-spacing:             close=c,crowded=w,distant=d
 *    8. gill-size:                broad=b,narrow=n
 *    9. gill-color:               black=k,brown=n,buff=b,chocolate=h,gray=g, green=r,orange=o,pink=p,purple=u,red=e, white=w,yellow=y
 *   10. stalk-shape:              enlarging=e,tapering=t
 *   11. stalk-root:               bulbous=b,club=c,cup=u,equal=e, rhizomorphs=z,rooted=r,missing=?
 *   12. stalk-surface-above-ring: fibrous=f,scaly=y,silky=k,smooth=s
 *   13. stalk-surface-below-ring: fibrous=f,scaly=y,silky=k,smooth=s
 *   14. stalk-color-above-ring:   brown=n,buff=b,cinnamon=c,gray=g,orange=o, pink=p,red=e,white=w,yellow=y
 *   15. stalk-color-below-ring:   brown=n,buff=b,cinnamon=c,gray=g,orange=o, pink=p,red=e,white=w,yellow=y
 *   16. veil-type:                partial=p,universal=u
 *   17. veil-color:               brown=n,orange=o,white=w,yellow=y
 *   18. ring-number:              none=n,one=o,two=t
 *   19. ring-type:                cobwebby=c,evanescent=e,flaring=f,large=l, none=n,pendant=p,sheathing=s,zone=z
 *   20. spore-print-color:        black=k,brown=n,buff=b,chocolate=h,green=r, orange=o,purple=u,white=w,yellow=y
 *   21. population:               abundant=a,clustered=c,numerous=n, scattered=s,several=v,solitary=y
 *   22. habitat:                  grasses=g,leaves=l,meadows=m,paths=p, urban=u,waste=w,woods=d

In [2]:
col_names = [
    "class","cap_shape","cap_surface","cap_color","bruises","odor","gill_attachment","gill_spacing","gill_size","gill_color","stalk_shape","stalk_root","stalk_surface_above_ring","stalk_surface_below_ring", 
    "stalk_color_above_ring","stalk_color_below_ring","veil_type","veil_color","ring_number","ring_type","spore_print_color","population","habitat"                   
]

# Data read from local upload
pdf = pd.read_csv("agaricus-lepiota.data", header = None, names = col_names)
pdf.head()

,class,cap_shape,cap_surface,cap_color,bruises,odor,gill_attachment,gill_spacing,gill_size,gill_color,...,stalk_surface_below_ring,stalk_color_above_ring,stalk_color_below_ring,veil_type,veil_color,ring_number,ring_type,spore_print_color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [3]:
# Converting to spark df
df = spark.createDataFrame(pdf)
df.printSchema()

root
 |-- class: string (nullable = true)
 |-- cap_shape: string (nullable = true)
 |-- cap_surface: string (nullable = true)
 |-- cap_color: string (nullable = true)
 |-- bruises: string (nullable = true)
 |-- odor: string (nullable = true)
 |-- gill_attachment: string (nullable = true)
 |-- gill_spacing: string (nullable = true)
 |-- gill_size: string (nullable = true)
 |-- gill_color: string (nullable = true)
 |-- stalk_shape: string (nullable = true)
 |-- stalk_root: string (nullable = true)
 |-- stalk_surface_above_ring: string (nullable = true)
 |-- stalk_surface_below_ring: string (nullable = true)
 |-- stalk_color_above_ring: string (nullable = true)
 |-- stalk_color_below_ring: string (nullable = true)
 |-- veil_type: string (nullable = true)
 |-- veil_color: string (nullable = true)
 |-- ring_number: string (nullable = true)
 |-- ring_type: string (nullable = true)
 |-- spore_print_color: string (nullable = true)
 |-- population: string (nullable = true)
 |-- habitat: string 

We can take a quick look at the distribution of our response variable `class`

In [4]:
df.groupBy("class").count().show()

+-----+-----+
|class|count|
+-----+-----+
|    e| 4208|
|    p| 3916|
+-----+-----+



We can see there are slightly more edible observations (51.7%) than poisonous (48.3%), but we still have a fairly even split, which is ideal.

We could also look at the distributions for all the other variables, but two in particular may be problematic:

In [5]:
df.groupBy("veil_type").count().show()

+---------+-----+
|veil_type|count|
+---------+-----+
|        p| 8124|
+---------+-----+



In [6]:
df.groupBy("stalk_root").count().show()

+----------+-----+
|stalk_root|count|
+----------+-----+
|         e| 1120|
|         c|  556|
|         b| 3776|
|         r|  192|
|         ?| 2480|
+----------+-----+



* `veil_type` only has one level (no variance), so this will be dropped in the pipeline
* the `stalk_root` variable has a `?` level representing missing data, with 2,480 counts. Since this is a lot of observations, we wont exclude these rows, and can keep this as a distinct 'missing' category later on.

## Train/Test Split

Let's setup the 80/20 split of our data:


In [7]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=7)
print(f"Training rows : {train_df.count()}")
print(f"Test rows     : {test_df.count()}")

Training rows : 6484
Test rows     : 1640


## Evaluation Metric - AUC

I'll use **AUC-ROC** (Area under receiver operating characteristic curve) for my primary metric as this will measure how well each model separates the class into edible vs poisonous. A naive classifier (guessing) would score around **0.5**, a perfect classifier scores **1**. This metric is better than raw accuracy in this context because it handles imbalances in our class (counts), and will help minimize false negatives (calling a poisonous mushroom edible). For interpretability though, we can also provide accuracy and f1 score.

In [8]:
# Import classification evaluators
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

We can also define a function to evaluate our models and return these metrics!

In [9]:
def evaluate(model, data, name=""):
    preds = model.transform(data)
    auc = evaluator_auc.evaluate(preds)
    acc = evaluator_acc.evaluate(preds)
    f1  = evaluator_f1.evaluate(preds)
    print(f"{'='*50}")
    print(f"  {name}")
    print(f"  AUC-ROC  : {auc:.4f}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  F1-Score : {f1:.4f}")
    print(f"{'='*50}")
    return {"model": name, "AUC": auc, "Accuracy": acc, "F1": f1}

## Data Preprocessing

Before selecting models to evaluate our data, we need to encode our predictor variables so they can be handled. The levels of each variable are displayed with letters, e.g: "e, c, b, r". These need to be indexed as "0, 1, 2, 3" for our response variable `class` and each predictor. We can do this using `StringIndexer`. Then, we can use `VectorAssembler` to combine all predictor columns into a single feature vector.

In [10]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, ChiSqSelector

# Drop veil_type since it has one category
feature_cols = [c for c in col_names if c not in ("class", "veil_type")]

# StringIndexer for each feature
feature_indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in feature_cols
]

# Label indexer: e -> 0, p -> 1 
label_indexer = StringIndexer(inputCol="class", outputCol="label")

indexed_cols = [c + "_idx" for c in feature_cols]

# VectorAssembler
assembler = VectorAssembler(inputCols=indexed_cols, outputCol="features")

Additionally, we can use `ChiSqSelector` to select top features to predict class by their chi-squared statistic. When we try a logistic regression model below, this will be **four transformations** within the pipeline (label indexer, feature indexer, vector assembler, chi square selector).

In [11]:
chi_selector = ChiSqSelector(
    numTopFeatures=15,
    featuresCol="features",
    outputCol="selected_features",
    labelCol="label"
)

## Model 1: Logistic Regression

First, we'll start with a logistic regression model. This is a linear classification model that estimates the probability a mushroom belongs to the poisonous class using **log-odds** (logit) of the outcome as a linear combination of predictor values. The model is fit by maximum likelihood, finding coefficients that make the observed data most probable under the logistic model.

In this model we'll use a regularation parameter (L2 penalty) to prevent overfitting

In [12]:
# Import pipeline, models for future, cv and parameter grid builder
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# LogReg Model
lr = LogisticRegression(
    featuresCol="selected_features",
    labelCol="label",
    maxIter=100
)

# Full pipeline with model inside
pipeline_lr = Pipeline(stages=[label_indexer] + feature_indexers + [assembler, chi_selector, lr])

# ParamGridBuilder() specifies possible tuning parameter values for our model
param_grid_lr = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0])
    .build()
)

# 5 fold cv, can specify auc metric with evaluarot
cv_lr = CrossValidator(
    estimator=pipeline_lr,
    estimatorParamMaps=param_grid_lr,
    evaluator=evaluator_auc,
    numFolds=5,
    seed=7
)

# Logistic regression model with 5 fold cv
cv_model_lr = cv_lr.fit(train_df)


26/04/13 21:15:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Additionally, we can save our best model to an object, and display the best tuning parameters.

In [13]:
best_lr = cv_model_lr.bestModel
lr_params = best_lr.stages[-1]
print(f"Best regParam     : {lr_params._java_obj.getRegParam()}") #Can access the best parameters with `_java_obj.`
print(f"Best elasticNet   : {lr_params._java_obj.getElasticNetParam()}")

Best regParam     : 0.01
Best elasticNet   : 0.0


## Model 2: Random Forest Classifier

Next, we'll fit a random forest classifier. This is an ensemble of **decision trees**. Each individual decision tree splits up the features using conditions, e.g. **is the odor_idx < 1?**. The final prediction is made by the majority vote of all the trees in the forest.

This model handles our categorical predictors without assuming any ordering, so splitting a variable will group the indeces into two sets.

In [14]:
# Random forest model
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    seed=7
)

# Full pipeline
pipeline_rf = Pipeline(stages=[label_indexer] + feature_indexers + [assembler, rf])

# Tuning parameter grid (reduced options to save runtime)
param_grid_rf = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50])
    .addGrid(rf.maxDepth, [5])
    .addGrid(rf.minInstancesPerNode, [1])
    .build()
)

# 5-fold cv
cv_rf = CrossValidator(
    estimator=pipeline_rf,
    estimatorParamMaps=param_grid_rf,
    evaluator=evaluator_auc,
    numFolds=5,
    seed=7
)

cv_model_rf = cv_rf.fit(train_df)

In [15]:
best_rf_model = cv_model_rf.bestModel.stages[-1]
print(f"Best numTrees           : {best_rf_model.getNumTrees}")
print(f"Best maxDepth           : {best_rf_model.getMaxDepth()}")
print(f"Best minInstancesPerNode: {best_rf_model.getMinInstancesPerNode()}")

Best numTrees           : 50
Best maxDepth           : 5
Best minInstancesPerNode: 1


## Model 3: Gradient Boosted Trees

Finally I'm trying a new model not used in this class yet called [Gradient Boosted Trees](https://spark.apache.org/docs/latest/ml-classification-regression.html#gradient-boosted-trees-gbts). This is similar to Random Forests above, but instead, decision trees are built sequentially. This means that instead of every decision being made independently, each new tree is trained to correct the residual error of the previous trees, in the direction of the **gradient** of the loss function.

Therefore, boosting directly targets mushrooms that were misclassified, and the final prediction is a weighted sum of all the trees outputs.

In [16]:
# GBT model
gbt = GBTClassifier(
    featuresCol="selected_features",
    labelCol="label",
    seed=42
)

# Pipeline
pipeline_gbt = Pipeline(stages=[label_indexer] + feature_indexers + [assembler, chi_selector, gbt])

# Parameter grid for tuning params (again reduced for runtime)
param_grid_gbt = (
    ParamGridBuilder()
    .addGrid(gbt.maxIter, [20])
    .addGrid(gbt.maxDepth, [3])
    .addGrid(gbt.stepSize, [0.1])
    .build()
)

# 5-fold cv
cv_gbt = CrossValidator(
    estimator=pipeline_gbt,
    estimatorParamMaps=param_grid_gbt,
    evaluator=evaluator_auc,
    numFolds=5,
    seed=7
)

cv_model_gbt = cv_gbt.fit(train_df)

In [17]:
best_gbt_model = cv_model_gbt.bestModel.stages[-1]
print(f"Best maxIter  : {best_gbt_model.getMaxIter()}")
print(f"Best maxDepth : {best_gbt_model.getMaxDepth()}")
print(f"Best stepSize : {best_gbt_model.getStepSize()}")

Best maxIter  : 20
Best maxDepth : 3
Best stepSize : 0.1


## Model Testing

Now, we'll evaluate the best model from each class (by AUC score) on the test data set.

In [18]:
results = []

results.append(evaluate(cv_model_lr,  test_df, "Logistic Regression (best CV model)"))
results.append(evaluate(cv_model_rf,  test_df, "Random Forest       (best CV model)"))
results.append(evaluate(cv_model_gbt, test_df, "Gradient Boosted Trees (best CV model)"))

  Logistic Regression (best CV model)
  AUC-ROC  : 0.9890
  Accuracy : 0.9646
  F1-Score : 0.9646
  Random Forest       (best CV model)
  AUC-ROC  : 1.0000
  Accuracy : 0.9994
  F1-Score : 0.9994
  Gradient Boosted Trees (best CV model)
  AUC-ROC  : 0.9997
  Accuracy : 0.9933
  F1-Score : 0.9933


In [19]:
results_df = pd.DataFrame(results).set_index("model")
print(results_df.to_string())

                                             AUC  Accuracy        F1
model                                                               
Logistic Regression (best CV model)     0.989037  0.964634  0.964621
Random Forest       (best CV model)     1.000000  0.999390  0.999390
Gradient Boosted Trees (best CV model)  0.999741  0.993293  0.993292


We can see that all three models perform very well at predicting edibility of mushrooms given these variables, with all three having AUC scores of over 0.98, and accuracy over 96%. F1 score is another metric that equally combines precision and recall (similar to type 1 and type 2 errors). In this context, this would be labeling an edible mushroom as poisonous, or labeling a poisonous mushroom as edible. The second is much worse, so this should probably be weighted more heavily.

The Random Forest model performed the best, with a perfect **1** AUC score, and 99.9% accuracy. It makes sense that this would outperform Logistic Regression due to it's complexity, but it's interesting that it slightly edged out the Gradient Boosted Trees. This could be because GBT slightly overfit the model or overcorrected mistakes of the first tree with its targeted boosting algorithm.

## Feature Importance

Finally, lets see which individual features are actually the most important, or best predictors of mushroom edibility in the random forest context. This gives us an idea of which predictors did the heavy lifting and should definitely be included in future models, and which are unimportant and could be left out.



In [20]:
rf_pipeline_model = cv_model_rf.bestModel
rf_stage = rf_pipeline_model.stages[-1]
importances = rf_stage.featureImportances.toArray()

feat_imp = pd.DataFrame({
    "feature": indexed_cols,
    "importance": importances
}).sort_values("importance", ascending=False).head(10)

print("Top 10 most important features:")
print(feat_imp.to_string(index=False))

Top 10 most important features:
                     feature  importance
                    odor_idx    0.385519
       spore_print_color_idx    0.125372
stalk_surface_above_ring_idx    0.098478
               gill_size_idx    0.085266
stalk_surface_below_ring_idx    0.075560
              gill_color_idx    0.057546
               ring_type_idx    0.033254
              stalk_root_idx    0.028739
              population_idx    0.025312
            gill_spacing_idx    0.023404


Here we can see the top 10 features by importance in the random forest model, with odor being the most important by a wide margin, followed by spore print color and stalk surface above ring. Here, an **importance** of 0.385 means that odor alone does roughly 38.5% of the predictive work in the random forest model, or 38.5% of decisions on mushroom edibility include mushroom odor. It reasons to believe all three of our models perform so well because odor alone has so much predictive power for edibility. We can also look at the distribution of edibility by odor to get a bigger picture.

In [24]:
df.groupBy("odor", "class").count().orderBy("odor").show()

+----+-----+-----+
|odor|class|count|
+----+-----+-----+
|   a|    e|  400|
|   c|    p|  192|
|   f|    p| 2160|
|   l|    e|  400|
|   m|    p|   36|
|   n|    e| 3408|
|   n|    p|  120|
|   p|    p|  256|
|   s|    p|  576|
|   y|    p|  576|
+----+-----+-----+



This table tells us all 400 'almond' odor mushrooms in this dataset were truly edible, all 192 'creosote' mushrooms were poisonous, all 2160 'foul' mushrooms were poisonous, and so on. All of the odor codes in our dataset except for non-odor (n = 3528) correspond to either edible or poisonous! This explains why if mushrooms had an odor, it was very easy for our models to accurately predict mushroom edibility, and it only really needed to use other variables when the mushroom had no odor.